# Project: A graph neural network from scratch 🕸️
Back to Zachary's karate club, but now with **only two labelled members**: the instructor (node 0) and the president (node 33). A 2-layer **graph convolutional network** (Kipf & Welling 2017) spreads those two labels through the friendship graph, and predicts everyone else's side. Write the normalised adjacency and the GCN layer yourself.

Topic: [[Graph Neural Networks]], [[Graph ML Overview]] · guide note: [[Project - GCN from Scratch]] · paper: [arXiv:1609.02907](https://arxiv.org/abs/1609.02907) · related: [[Project - Spectral Clustering of the Karate Club]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, networkx as nx, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
G = nx.karate_club_graph(); A = torch.tensor(nx.to_numpy_array(G, weight=None), dtype=torch.float32)
truth = torch.tensor([0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in G]); n = len(truth)
X = torch.eye(n)                                           # no features: each node is a one-hot ID (the graph does the work)
train_idx = torch.tensor([0, 33])

## 1. Renormalised adjacency\n$\hat A = A + I$ (self-loops, so a node keeps its own features), then $\tilde A = \hat D^{-1/2}\hat A\hat D^{-1/2}$ with $\hat D$ the degree matrix of $\hat A$.

In [ ]:
def normalize_adjacency(A):
    # TODO 1: add self-loops, symmetric degree normalisation
    raise NotImplementedError  # TODO 1

## 2. The GCN layer\n$H' = \tilde A\,H\,W$: transform every node's features with $W$, then average over neighbours (weighted by $\tilde A$). The non-linearity is applied outside.

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__(); self.W = nn.Parameter(torch.empty(d_in, d_out)); nn.init.xavier_uniform_(self.W)
    def forward(self, H, A_norm):
        # TODO 2: neighbourhood aggregation of the transformed features
        raise NotImplementedError  # TODO 2

In [ ]:
def _t_gcn():
    A2 = torch.tensor([[0, 1.0], [1, 0]]); An = normalize_adjacency(A2)
    layer = GCNLayer(2, 3); H = torch.randn(2, 2)
    return torch.allclose(An, torch.full((2, 2), 0.5)) and torch.allclose(layer(H, An), An @ H @ layer.W)
check("GCN parts", _t_gcn, "Two connected nodes: Â = [[1,1],[1,1]], degrees 2 → every entry 1/2; the layer is Ã H W.")
if ready("GCN parts"):
    A_norm = normalize_adjacency(A)
    class GCN(nn.Module):
        def __init__(self): super().__init__(); self.l1, self.l2 = GCNLayer(n, 16), GCNLayer(16, 2)
        def forward(self, X, A_norm): return self.l2(F.relu(self.l1(X, A_norm)), A_norm)
    accs = []
    for seed in range(5):
        torch.manual_seed(seed); model = GCN(); opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
        for epoch in range(200):
            loss = F.cross_entropy(model(X, A_norm)[train_idx], truth[train_idx]); opt.zero_grad(); loss.backward(); opt.step()
        pred = model(X, A_norm).argmax(1); accs.append((pred == truth).float().mean().item())
    print(f"accuracy on all 34 members from 2 labels, 5 seeds: {np.round(accs, 3)} (mean {np.mean(accs):.1%})")
    H = F.relu(model.l1(X, A_norm)).detach().numpy(); pos = nx.spring_layout(G, seed=3)
    nx.draw(G, pos, node_color=pred.numpy(), cmap="coolwarm", with_labels=True, font_size=7, node_size=[400 if i in (0, 33) else 150 for i in G]); plt.title("GCN predictions (big nodes = the only 2 labels)"); plt.show()
    check("semi-supervised GCN", lambda: np.mean(accs) > 0.85, "From two labels the GCN should classify > 85 % of members correctly on average.")

<details><summary>▶ Solution</summary>

```python
def normalize_adjacency(A):
    A_hat = A + torch.eye(len(A))
    d = A_hat.sum(1)
    D = torch.diag(d.pow(-0.5))
    return D @ A_hat @ D
```

```python
    def forward(self, H, A_norm):   # GCNLayer
        return A_norm @ H @ self.W
```
</details>

## Stretch goals
- Remove the self-loops (use plain $D^{-1/2}AD^{-1/2}$). What changes?
- Stack 2, 4, 8 layers: watch **over-smoothing** make all node embeddings the same.
- Try the Cora citation dataset (via `torch_geometric`), the paper's benchmark.